# PricePoint — 01 · Data Preprocessing

Turn the raw listing file `data/laptop_price.csv` into a clean table the models can use.

**What we do, step by step:**
1. Load the CSV
2. Extract screen info (resolution pixels, touchscreen, IPS)
3. Extract CPU brand and speed
4. Extract total storage size and storage type (SSD / HDD / Hybrid)
5. Extract GPU brand
6. Map the OS to simple names
7. Clean RAM and weight (remove "GB" / "kg")
8. Convert euro price to BDT
9. Put everything into one clean table and save it

In [1]:
import re

import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# --- file paths -------------------------------------------------------
# If we are running from the notebooks/ folder, data is one level up.
# If we are running from the ml/ folder, data is right here.

if Path.cwd().name == "notebooks":
    DATA_DIR = Path("../data")
else:
    DATA_DIR = Path("data")

RAW_PATH = DATA_DIR / "laptop_price.csv"
CLEAN_PATH = DATA_DIR / "laptop_price_clean.csv"

# exchange rate: 1 euro = how many taka
EUR_TO_BDT = 125.0

print("Reading from:", RAW_PATH)

Reading from: ../data/laptop_price.csv


In [3]:
# --- 1. load the raw data --------------------------------------------

df = pd.read_csv(RAW_PATH, encoding="latin-1")

print("rows and columns:", df.shape)
print("missing values:", df.isna().sum().sum())
df.head()

rows and columns: (1303, 13)
missing values: 0


,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


## 2. Screen features

The `ScreenResolution` column looks like:
`"IPS Panel Retina Display 2560x1600"`

We pull out:
- the two numbers (`2560` and `1600`) so we can multiply them into total pixels
- whether it says **Touchscreen**
- whether it says **IPS**

In [4]:
# split "2560x1600" into two numbers
resolution_parts = df["ScreenResolution"].str.extract(r"(\d{3,4})x(\d{3,4})")

screen_width = pd.to_numeric(resolution_parts[0])
screen_height = pd.to_numeric(resolution_parts[1])

resolution_pixels = (screen_width * screen_height).astype(int)

# True / False for these two features
touchscreen = df["ScreenResolution"].str.contains("Touchscreen")
ips = df["ScreenResolution"].str.contains("IPS")

print("pixels range:", resolution_pixels.min(), "to", resolution_pixels.max())
print("touchscreen laptops:", touchscreen.sum())
print("IPS laptops:", ips.sum())

pixels range: 1049088 to 8294400
touchscreen laptops: 192
IPS laptops: 365


## 3. CPU features

`Cpu` looks like: `"Intel Core i5 7200U 2.5GHz"`

We want:
- **cpu_brand** → `"Intel i5"`, `"Intel i7"`, `"Intel i3"`, `"AMD"`, ... (just use simple if / elif)
- **cpu_speed_ghz** → the number before `GHz`

In [5]:
def get_cpu_brand(cpu_text):
    if "Intel Core i7" in cpu_text:
        return "Intel i7"
    elif "Intel Core i5" in cpu_text:
        return "Intel i5"
    elif "Intel Core i3" in cpu_text:
        return "Intel i3"
    elif "Intel" in cpu_text:
        return "Intel Other"
    elif "AMD" in cpu_text:
        return "AMD"
    else:
        return "Other"


def get_cpu_speed(cpu_text):
    # find the number right before "GHz", e.g. "2.5GHz" -> 2.5
    match = re.search(r"([\d.]+)GHz", cpu_text)
    if match:
        return float(match.group(1))
    return 0.0


cpu_brand = df["Cpu"].apply(get_cpu_brand)
cpu_speed_ghz = df["Cpu"].apply(get_cpu_speed)

print(cpu_brand.value_counts())

Cpu
Intel i7       527
Intel i5       423
Intel Other    154
Intel i3       136
AMD             62
Other            1
Name: count, dtype: int64


## 4. Storage (Memory)

`Memory` looks like:
- `"128GB SSD"`
- `"1TB HDD + 128GB SSD"`  (a laptop with two drives)

We want:
- **storage_gb** → total size in GB (1TB = 1024GB)
- **storage_type** → `"SSD"`, `"HDD"`, or `"Hybrid"` (both)

In [6]:
def get_storage(memory_text):
    # find every "256GB SSD", "1TB HDD", ... part in the text
    parts = re.findall(r"(\d+)(GB|TB)\s+([A-Za-z ]+)", memory_text)

    total_gb = 0
    has_ssd = False
    has_hdd = False

    for number, unit, drive_type in parts:
        number = int(number)
        if unit == "TB":
            total_gb += number * 1024
        else:
            total_gb += number

        if "SSD" in drive_type or "Flash" in drive_type:
            has_ssd = True
        elif "HDD" in drive_type:
            has_hdd = True

    # decide the final storage type
    if has_ssd and has_hdd:
        storage_type = "Hybrid"
    elif has_ssd:
        storage_type = "SSD"
    elif has_hdd:
        storage_type = "HDD"
    else:
        storage_type = "Other"

    return total_gb, storage_type


storage_info = df["Memory"].apply(get_storage)

storage_gb = storage_info.apply(lambda x: x[0])
storage_type = storage_info.apply(lambda x: x[1])

print(storage_type.value_counts())

Memory
SSD       717
HDD       375
Hybrid    201
Other      10
Name: count, dtype: int64


## 5. GPU brand

`Gpu` looks like `"Nvidia GeForce MX150"` or `"Intel HD Graphics 620"`.

We only care about **who makes it**: Intel, Nvidia, or AMD.

In [7]:
def get_gpu_brand(gpu_text):
    if "Intel" in gpu_text:
        return "Intel"
    elif "Nvidia" in gpu_text:
        return "Nvidia"
    elif "AMD" in gpu_text:
        return "AMD"
    else:
        return "Other"


gpu_brand = df["Gpu"].apply(get_gpu_brand)

print(gpu_brand.value_counts())

Gpu
Intel     722
Nvidia    400
AMD       180
Other       1
Name: count, dtype: int64


## 6. OS

`OpSys` has many similar values (`Windows 10`, `Windows 10 S`, `Windows 7`, ...).

We group them into simple names with a dictionary.

In [8]:
os_lookup = {
    "Windows 10": "Windows",
    "Windows 10 S": "Windows",
    "Windows 7": "Windows",
    "macOS": "Mac",
    "Mac OS X": "Mac",
    "Linux": "Linux",
    "Chrome OS": "Chrome OS",
    "Android": "Android",
    "No OS": "No OS",
}

os_name = df["OpSys"].map(lambda x: os_lookup.get(x, "Other"))

print(os_name.value_counts())

OpSys
Windows      1125
No OS          66
Linux          62
Chrome OS      27
Mac            21
Android         2
Name: count, dtype: int64


## 7. Clean RAM, weight, and screen size

Just remove the letters and turn them into numbers:
- `"8GB"` → `8`
- `"1.37kg"` → `1.37`

In [9]:
ram_gb = df["Ram"].str.replace("GB", "", regex=False).astype(int)
weight_kg = df["Weight"].str.replace("kg", "", regex=False).astype(float)
inches = df["Inches"].astype(float)

print("RAM values:", ram_gb.unique())
print("weight min/max:", weight_kg.min(), weight_kg.max())

RAM values: [ 8 16  4  2 12  6 32 24 64]
weight min/max: 0.69 4.7


## 8. Convert the price from euros to taka (BDT)

`price_bdt = price_euros × 125`

In [10]:
price_euros = df["Price_euros"].astype(float)
price_bdt = (price_euros * EUR_TO_BDT).round().astype(int)

print("min price:", price_bdt.min(), "BDT")
print("median price:", price_bdt.median(), "BDT")
print("max price:", price_bdt.max(), "BDT")

min price: 21750 BDT
median price: 122125.0 BDT
max price: 762375 BDT


## 9. Put it all together into one clean table

In [11]:
clean = pd.DataFrame({
    "brand": df["Company"],
    "type": df["TypeName"],
    "inches": inches,
    "touchscreen": touchscreen,
    "ips": ips,
    "resolution_pixels": resolution_pixels,
    "cpu_brand": cpu_brand,
    "cpu_speed_ghz": cpu_speed_ghz,
    "ram_gb": ram_gb,
    "storage_gb": storage_gb,
    "storage_type": storage_type,
    "gpu_brand": gpu_brand,
    "os": os_name,
    "weight_kg": weight_kg,
    "price_euros": price_euros,
    "price_bdt": price_bdt,
})

clean.head()

,brand,type,inches,touchscreen,ips,resolution_pixels,cpu_brand,cpu_speed_ghz,ram_gb,storage_gb,storage_type,gpu_brand,os,weight_kg,price_euros,price_bdt
0,Apple,Ultrabook,13.3,False,True,4096000,Intel i5,2.3,8,128,SSD,Intel,Mac,1.37,1339.69,167461
1,Apple,Ultrabook,13.3,False,False,1296000,Intel i5,1.8,8,128,SSD,Intel,Mac,1.34,898.94,112368
2,HP,Notebook,15.6,False,False,2073600,Intel i5,2.5,8,256,SSD,Intel,No OS,1.86,575.00,71875
3,Apple,Ultrabook,15.4,False,True,5184000,Intel i7,2.7,16,512,SSD,AMD,Mac,1.83,2537.45,317181
4,Apple,Ultrabook,13.3,False,True,4096000,Intel i5,3.1,8,256,SSD,Intel,Mac,1.37,1803.60,225450


## 10. Save the clean file and show a quick summary

In [12]:
clean.to_csv(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH)
print("rows:", len(clean))
print()

# quick summary of the important categories
for column in ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]:
    print(column + ":")
    print(clean[column].value_counts().to_string())
    print()

Saved: ../data/laptop_price_clean.csv
rows: 1303

brand:
brand
Dell         297
Lenovo       297
HP           274
Asus         158
Acer         103
MSI           54
Toshiba       48
Apple         21
Samsung        9
Razer          7
Mediacom       7
Microsoft      6
Xiaomi         4
Vero           4
Chuwi          3
Google         3
Fujitsu        3
LG             3
Huawei         2

type:
type
Notebook              727
Gaming                205
Ultrabook             196
2 in 1 Convertible    121
Workstation            29
Netbook                25

cpu_brand:
cpu_brand
Intel i7       527
Intel i5       423
Intel Other    154
Intel i3       136
AMD             62
Other            1

storage_type:
storage_type
SSD       717
HDD       375
Hybrid    201
Other      10

gpu_brand:
gpu_brand
Intel     722
Nvidia    400
AMD       180
Other       1

os:
os
Windows      1125
No OS          66
Linux          62
Chrome OS      27
Mac            21
Android         2

